# Smoothed-Hinge Block Coordinate Descent — Breast Cancer


In [ ]:
import numpy as np
import time
import matplotlib
import matplotlib.pyplot as plt
from numba import njit, prange
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer

matplotlib.rcParams['text.usetex'] = False  # avoid 'latex not found'

In [ ]:
def load_and_convert_dataset():
    """Breast cancer data, labels in {-1,+1}, standardized features."""
    cancer = load_breast_cancer()
    A = cancer.data
    B = 2 * cancer.target - 1            # {0,1} -> {-1,+1}
    A = StandardScaler().fit_transform(A)
    return A, B.astype(np.float64)

In [ ]:
def step_sizes(a, block_vector, length_vector, factor=1.9):
    """Per-case step size 1.9 / max_block ||a_block||_2^2."""
    cases = len(block_vector)
    out = np.zeros(cases)
    for i in range(cases):
        block, length = int(block_vector[i]), int(length_vector[i])
        step = 0.0
        for l in range(block):
            a_block = a[:, l*length:(l+1)*length]
            step = max(step, np.linalg.norm(a_block, 2)**2)
        out[i] = factor / step
    return out

In [ ]:
# ===================== Smoothed hinge phi_alpha (CORRECTED) =====================
# phi(t) = 0 (t<=0); 0.5 t^2 (0<t<=1); (t^a - 1)/a + 0.5 (t>1)   -- C^1, continuous.

@njit(cache=True, fastmath=True)
def falpha(t, alpha, d):
    s = 0.0
    for i in range(d):
        ti = t[i]
        if ti <= 0.0:
            continue
        elif ti <= 1.0:
            s += 0.5 * ti * ti
        else:
            s += (ti**alpha - 1.0) / alpha + 0.5     # corrected: -1, not -alpha
    return s

@njit(cache=True, fastmath=True)
def deriv_phi(t, b, alpha, d, out):
    """out[i] = -b[i] * phi'(t[i]); written into preallocated `out`."""
    for i in range(d):
        ti = t[i]
        if ti <= 0.0:
            out[i] = 0.0
        elif ti <= 1.0:
            out[i] = -b[i] * ti
        else:
            out[i] = -b[i] * ti**(alpha - 1.0)

In [ ]:
@njit(cache=True, fastmath=True)
def coordinate_descent(a, b, alpha, p, d, block, length,
                       num_iterations, x0, random_blocks, lr):
    """Block CD (contiguous blocks) with in-place incremental t-update."""
    F_hist = np.zeros(num_iterations + 1)
    x = x0.copy()
    t = np.empty(d)
    for i in range(d):
        s = 0.0
        for j in range(p):
            s += a[i, j] * x[j]
        t[i] = 1.0 - b[i] * s

    deriv = np.empty(d)
    F_hist[0] = falpha(t, alpha, d)
    total = num_iterations * block
    for it in range(total):
        start = random_blocks[it] * length
        deriv_phi(t, b, alpha, d, deriv)
        for jj in range(length):
            col = start + jj
            g = 0.0
            for i in range(d):
                g += a[i, col] * deriv[i]
            dx = -lr * g
            x[col] += dx
            for i in range(d):                # rank-1 incremental t update
                t[i] -= b[i] * a[i, col] * dx
        if it % block == 0:
            F_hist[it // block + 1] = falpha(t, alpha, d)
    return x, F_hist


@njit(cache=True, fastmath=True)
def coordinate_descent_rand(a, b, alpha, p, d, block, length,
                            num_iterations, x0, random_blocks, lr):
    """Block CD with arbitrary (random, no-replacement) column subsets.
    random_blocks: (num_iterations*block, length) int64 of global column indices."""
    F_hist = np.zeros(num_iterations + 1)
    x = x0.copy()
    t = np.empty(d)
    for i in range(d):
        s = 0.0
        for j in range(p):
            s += a[i, j] * x[j]
        t[i] = 1.0 - b[i] * s

    deriv = np.empty(d)
    F_hist[0] = falpha(t, alpha, d)
    total = num_iterations * block
    for it in range(total):
        blk = random_blocks[it]
        deriv_phi(t, b, alpha, d, deriv)
        for jj in range(length):
            col = blk[jj]
            g = 0.0
            for i in range(d):
                g += a[i, col] * deriv[i]
            dx = -lr * g
            x[col] += dx
            for i in range(d):
                t[i] -= b[i] * a[i, col] * dx
        if it % block == 0:
            F_hist[it // block + 1] = falpha(t, alpha, d)
    return x, F_hist

In [ ]:
def run_cd_mc(a, b, alpha, p, d, block_vector, length_vector,
              lr_vector, MC, num_iterations, Init, rng, random_subset=False):
    """Monte-Carlo driver for both contiguous and random-subset CD."""
    cases = len(block_vector)
    x_train = np.zeros((p, cases, MC))
    F_CD    = np.zeros((cases, num_iterations + 1, MC))
    CD_time = np.zeros((cases, MC))
    for k in range(cases):
        block, length, lr = int(block_vector[k]), int(length_vector[k]), lr_vector[k]
        print(f"Case {k}: blocks={block}, length={length}")
        for mc in range(MC):
            x0 = Init[:, mc].copy()
            if random_subset:
                # batched argsort: fast no-replacement subsets of size `length`
                rb = np.argsort(rng.random((num_iterations*block, p)), axis=1)[:, :length].astype(np.int64)
                run = coordinate_descent_rand
            else:
                rb = rng.integers(0, block, size=num_iterations*block).astype(np.int64)
                run = coordinate_descent
            t0 = time.time()
            xf, Fh = run(a, b, alpha, p, d, block, length, num_iterations, x0, rb, lr)
            CD_time[k, mc] = time.time() - t0
            x_train[:, k, mc] = xf
            F_CD[k, :, mc] = Fh
    return x_train, F_CD, CD_time

In [ ]:
# ===================== Load data & parameters =====================
A, B = load_and_convert_dataset()
dim = 569
a_train, b_train = A[:dim], B[:dim]
a_test,  b_test  = A[dim:], B[dim:]
a, b = a_train, b_train
d, p = a.shape

alpha = 0.1
MC_all, iter_all = 10, 200000
block_vector  = [1, 2, 3, 5, 6, 10, 15, 30]
length_vector = [30, 15, 10, 6, 5, 3, 2, 1]
MC, num_iterations = MC_all, iter_all

rng = np.random.default_rng(123)
Init = rng.normal(0, 1, (p, MC))
lr_vector = step_sizes(a, block_vector, length_vector, factor=1.9)
print("step sizes:", lr_vector)

step sizes: [0.00025141 0.00045044 0.00058611 0.00079512 0.00089515 0.00131206
 0.00168094 0.00333919]


In [ ]:
# ===================== Run contiguous-block CD =====================
x_train, F_CD, CD_time = run_cd_mc(
    a, b, alpha, p, d, block_vector, length_vector,
    lr_vector, MC, num_iterations, Init, rng, random_subset=False)
print("avg time per case:", np.mean(CD_time, axis=1))

Case 0: blocks=1, length=30


In [ ]:
# ===================== Run random-subset CD =====================
x_train_rand, F_CD_rand, CD_time_rand = run_cd_mc(
    a, b, alpha, p, d, block_vector, length_vector,
    lr_vector, MC, num_iterations, Init, rng, random_subset=True)
print("avg time per case:", np.mean(CD_time_rand, axis=1))

In [ ]:
def plot_loss_functions(start, end, F_CD, block_vector, save_path="cd.jpeg"):
    iterations = np.arange(start + 1, end)
    colors = ['aqua','maroon','fuchsia','orange','green','gold','pink','black']
    plt.figure(figsize=(10, 6))
    for c, blk in enumerate(block_vector):
        if c >= len(colors): break
        loss = F_CD[c, start:end, :]
        m, s = loss.mean(1), loss.std(1)
        plt.plot(iterations, m, label=f"Blocks {blk}", color=colors[c], lw=2)
        plt.fill_between(iterations, m - s, m + s, color=colors[c], alpha=0.2)
    plt.xscale('log'); plt.yscale('log')
    plt.title('Loss over iterations'); plt.xlabel('Iteration'); plt.ylabel('Averaged loss')
    plt.legend(fontsize='small'); plt.tight_layout()
    plt.savefig(save_path, dpi=150); plt.show()

plot_loss_functions(1, num_iterations + 2, F_CD, block_vector, "cd_contiguous.jpeg")

In [ ]:
# Test/train objective per case (sanity)
for kk in range(len(block_vector)):
    x1 = x_train[:, kk, MC-1]
    t_tr = 1 - b_train * (a_train @ x1)
    t_te = 1 - b_test  * (a_test  @ x1)
    print(f"case {kk}: train={falpha(t_tr, alpha, dim):.4f}  "
          f"test={falpha(t_te, alpha, len(b_test)):.4f}")